In [5]:
import pandas as pd
import numpy as np
import scipy.io as sio

In [6]:
# Do for EDML
# Load MATLAB .mat and convert MATLAB structs/cell arrays to numpy-friendly structures
mat_path = '/Users/quinnmackay/Documents/GitHub/BICC_Holocene_LayerCount/matchfiles/EDML_match.mat'
mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
# show top-level keys
print('EDML MAT keys:', [k for k in mat.keys() if not k.startswith('__')])

def _todict(matobj):
    from collections import OrderedDict
    out = OrderedDict()
    for field in matobj._fieldnames:
        val = getattr(matobj, field)
        if hasattr(val, '_fieldnames'):
            out[field] = _todict(val)
        elif isinstance(val, (list, tuple)):
            out[field] = [_todict(x) if hasattr(x, '_fieldnames') else x for x in val]
        elif hasattr(val, 'dtype') and val.dtype == object:
            out[field] = [_todict(x) if hasattr(x, '_fieldnames') else x for x in val]
        else:
            out[field] = val
    return out

# Convert top-level entries into numpy-friendly python objects
edml_data = {}
for k, v in mat.items():
    if k.startswith('__'):
        continue
    if hasattr(v, '_fieldnames'):
        edml_data[k] = _todict(v)
    elif hasattr(v, 'dtype') and v.dtype == object:
        edml_data[k] = [(_todict(x) if hasattr(x, '_fieldnames') else x) for x in v]
    else:
        edml_data[k] = v

# Do for WDC
# Load MATLAB .mat and convert MATLAB structs/cell arrays to numpy-friendly structures
import scipy.io as sio
mat_path = '/Users/quinnmackay/Documents/GitHub/BICC_Holocene_LayerCount/matchfiles/WDC_match.mat'
mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
# show top-level keys
print('WDC MAT keys:', [k for k in mat.keys() if not k.startswith('__')])

def _todict(matobj):
    from collections import OrderedDict
    out = OrderedDict()
    for field in matobj._fieldnames:
        val = getattr(matobj, field)
        if hasattr(val, '_fieldnames'):
            out[field] = _todict(val)
        elif isinstance(val, (list, tuple)):
            out[field] = [_todict(x) if hasattr(x, '_fieldnames') else x for x in val]
        elif hasattr(val, 'dtype') and val.dtype == object:
            out[field] = [_todict(x) if hasattr(x, '_fieldnames') else x for x in val]
        else:
            out[field] = val
    return out

# Convert top-level entries into numpy-friendly python objects
wdc_data = {}
for k, v in mat.items():
    if k.startswith('__'):
        continue
    if hasattr(v, '_fieldnames'):
        wdc_data[k] = _todict(v)
    elif hasattr(v, 'dtype') and v.dtype == object:
        wdc_data[k] = [(_todict(x) if hasattr(x, '_fieldnames') else x) for x in v]
    else:
        wdc_data[k] = v

EDML MAT keys: ['mp']
WDC MAT keys: ['mp']


In [7]:
# recreate the normal arrays

#conversion from matchmaker counts to vals
def convert_count(mm_depth, mm_count):
    count = []
    depths = []
    for d, c in zip(mm_depth, mm_count):
        if c == 1:
            continue
        if c == 6:
            count.append(1)
            depths.append(d)
        elif c == 7:
            count.append(1)
            depths.append(d)

    return depths, count


#1 is volc, 6 is normal, 7 is considered UNCERTAIN, not half

#EDML first
original_edml_lc = pd.read_csv('EDML_LayerCount.txt', sep='\t')
starting_year_edml = original_edml_lc['Age'][0]

mm_edml_depths = edml_data['mp'][:,0]
mm_edml_count = edml_data['mp'][:,1]

edml_lc_modified = pd.DataFrame()
edml_depth, edml_count = convert_count(mm_edml_depths, mm_edml_count)
edml_lc_modified['Depth'] = edml_depth
edml_lc_modified['Count'] = edml_count
edml_lc_modified['Age'] = np.zeros(len(edml_lc_modified))

#assign ages
edml_lc_modified.loc[0, 'Age'] = starting_year_edml
for i in range(1, len(edml_lc_modified)):
    edml_lc_modified.loc[i, 'Age'] = edml_lc_modified['Age'][i-1] + edml_lc_modified['Count'][i]

edml_lc_modified = edml_lc_modified[edml_lc_modified['Age'] <= 11999]

#WD next
original_wdc_lc = pd.read_csv('WDC_LayerCount.txt', sep='\t')
starting_year_wdc = original_wdc_lc['Age'][0]

mm_wdc_depths = wdc_data['mp'][:,0]
mm_wdc_count = wdc_data['mp'][:,1]

wdc_lc_modified = pd.DataFrame()
wdc_depth, wdc_count = convert_count(mm_wdc_depths, mm_wdc_count)
wdc_lc_modified['Depth'] = wdc_depth
wdc_lc_modified['Count'] = wdc_count 
wdc_lc_modified['Age'] = np.zeros(len(wdc_lc_modified))

#assign ages
wdc_lc_modified.loc[0, 'Age'] = starting_year_wdc
for i in range(1, len(wdc_lc_modified)):
    wdc_lc_modified.loc[i, 'Age'] = wdc_lc_modified['Age'][i-1] + wdc_lc_modified['Count'][i]

wdc_lc_modified = wdc_lc_modified[wdc_lc_modified['Age'] <= 11999]

wdc_lc_modified = wdc_lc_modified[['Depth', 'Age']] #drop the count because not used in next chunk


In [8]:
# create compare excel sheet

#load LCs

edml_lc = edml_lc_modified
wdc_lc = wdc_lc_modified

#load tiepoints
match_name = 'EDML-WDC'
core1=match_name.split('-')[0]
core2=match_name.split('-')[1]

big_table = pd.read_csv('/Users/quinnmackay/Documents/GitHub/IceTiepoint_Analysis/Network Analysis/big_table/all_tiepoints_big_table.csv')

compare = big_table[[f'{match_name}_{core1}', f'{match_name}_{core2}']].dropna()

#drop any rows with tiepoints lower than LC max depths
compare = compare[compare[f'{match_name}_{core1}'] <= edml_lc['Depth'].max()]
compare = compare[compare[f'{match_name}_{core2}'] <= wdc_lc['Depth'].max()]

compare.rename(columns={f'{match_name}_{core1}': f'{core1} m', f'{match_name}_{core2}': f'{core2} m'}, inplace=True)
compare.sort_values(by=[f'{core1} m'], inplace=True)
compare.reset_index(drop=True, inplace=True)

#get the ages from the tiepoints
compare[f'{core1} age'] = np.interp(compare[f'{core1} m'], edml_lc['Depth'], edml_lc['Age'])
compare[f'{core2} age'] = np.interp(compare[f'{core2} m'], wdc_lc['Depth'], wdc_lc['Age'])

#calc discrepancies
compare[f'difference ({core1}-{core2})'] = compare[f'{core1} age'] - compare[f'{core2} age'] # NEGATIVE means EDML undercounted relative
compare

#calc section error

compare[f'section error'] = compare[f'difference ({core1}-{core2})'].diff()
compare.loc[0, 'section error'] = 0
compare.to_excel('Modified_EDML-WDC_tiepoint_comparison.xlsx', index=False)

/var/folders/pq/tk8k94gn22v6mr07byx05hzc0000gn/T/ipykernel_85620/2307296826.py:13: DtypeWarning: Columns (2,3,6,7,10,11,14,15,18,19,22,23,26,27,30,31,34,35,38,39,42,43,46,47,50,51,54,55,58,59,62,63,66,67,70,71,74,75,78,79,82,83,86,87,90,91,94,95,98,99,102,103,106,107,110,111,114,115,118,119,122,123,126,127,130,131,134,135,138,139,142,143,146,147,150,151,154,155,158,159,162,163) have mixed types. Specify dtype option on import or set low_memory=False.
  big_table = pd.read_csv('/Users/quinnmackay/Documents/GitHub/IceTiepoint_Analysis/Network Analysis/big_table/all_tiepoints_big_table.csv')
